# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulmoeed1090/Fly-rank-starter-assignment/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The queue maps model-predicted probabilities of future impression decline into human-actionable recommendations.

Each item in the review queue receives an explicit, deterministic Reason Code based on current-day search signals and temporal trend features:

- `POS_SLIP`: Average search position degraded compared to the 1-day/2-day historical baseline (position_delta > 2.0).

- `IMP_DROP_VELOCITY`: Impressions declined sharply relative to the 3-day rolling average (impression_vs_3d_avg < 0.75).

- `LOW_CTR_HIGH_IMP`: High impression volume with sub-optimal click-through rate (ctr < 0.01 and gsc_impressions >= 20).

- `DECAY_RISK`: General multi-period momentum loss without a single dominant anomaly.

This mapping translates raw ML probabilities into transparent human-trusted audit candidate categories.

In [1]:
# Imports and environment setup
import os
import json
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.ensemble import HistGradientBoostingClassifier

print("Imports completed.")

# Loading dataset efficiently to prevent memory/CPU hangs
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

# Extract 50,000 rows directly at the Arrow level before converting to Pandas
train_slice = dataset["train"].select(range(min(50000, len(dataset["train"]))))
df = train_slice.to_pandas()

print("Dataset shape:", df.shape)
print("Date range:", df["report_date"].min(), "to", df["report_date"].max())

# Basic preparation
df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

# CTR construction
df["ctr"] = (df["gsc_clicks"] / df["gsc_impressions"].replace(0, 1)).clip(0, 1)

print("\nFeature Summary:")
print(df[feature_cols + ["ctr"]].describe())

Imports completed.


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Dataset shape: (50000, 30)
Date range: 2025-01-27 to 2025-02-27

Feature Summary:
       gsc_impressions    gsc_clicks  gsc_avg_position           ctr
count     50000.000000  50000.000000      50000.000000  50000.000000
mean         15.461200      0.105300         28.559778      0.007629
std          25.695012      0.451637         22.826635      0.048061
min           1.000000      0.000000          0.000000      0.000000
25%           3.000000      0.000000          9.000000      0.000000
50%           8.000000      0.000000         21.666667      0.000000
75%          18.000000      0.000000         43.000000      0.000000
max         818.000000     16.000000        141.000000      1.000000


## 2. Intended use and limits

Intended Use:Editorial Prioritization: Serves as a decision-support filter for content managers to identify high-traffic pages exhibiting performance decay.Review Queue Filtering: Focuses auditing resources on the top $K$ pages (e.g., Precision@20) rather than randomly scanning thousands of URLs.Operational Boundaries & Validity Limits:Does NOT Prove Cause: High decline probability indicates observed statistical risk, not that content quality is poor or that refreshing will guarantee traffic recovery.Micro-Scale Noise: Not valid for low-traffic pages ($< 5$ daily impressions) where day-to-day variance dominates signal.Seasonality & External Shifts: Cannot account for major Google search algorithm updates, search intent shifts, or seasonal industry traffic drops.

In [3]:
# Imports and environment setup
import os
import json
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.ensemble import HistGradientBoostingClassifier

print("Imports completed.")

# 1. Load warehouse data slice efficiently (avoids memory hangs)
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

train_slice = dataset["train"].select(range(min(50000, len(dataset["train"]))))
df = train_slice.to_pandas()

df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")
feature_cols = ["gsc_impressions", "gsc_clicks", "gsc_avg_position"]
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df["ctr"] = (df["gsc_clicks"] / df["gsc_impressions"].replace(0, 1)).clip(0, 1)

# Feature engineering matching ML-08/ML-09
df = df.sort_values(["client_hash_id", "content_hash_id", "report_date"]).reset_index(drop=True)
group_cols = ["client_hash_id", "content_hash_id"]

df["prev_impressions"] = df.groupby(group_cols)["gsc_impressions"].shift(1).fillna(df["gsc_impressions"])
df["impression_delta"] = df["gsc_impressions"] - df["prev_impressions"]
df["impression_ratio"] = (df["gsc_impressions"] + 1) / (df["prev_impressions"] + 1)

df["prev_position"] = df.groupby(group_cols)["gsc_avg_position"].shift(1).fillna(df["gsc_avg_position"])
df["position_delta"] = df["gsc_avg_position"] - df["prev_position"]

df["prev2_impressions"] = df.groupby(group_cols)["gsc_impressions"].shift(2).fillna(df["prev_impressions"])
df["impression_delta_2d"] = df["gsc_impressions"] - df["prev2_impressions"]

df["rolling_3d_impressions"] = df.groupby(group_cols)["gsc_impressions"].transform(lambda x: x.rolling(3, min_periods=1).mean())
df["impression_vs_3d_avg"] = (df["gsc_impressions"] + 1) / (df["rolling_3d_impressions"] + 1)

# Target setup
df["future_date"] = df.groupby(group_cols)["report_date"].shift(-1)
df["future_impressions"] = df.groupby(group_cols)["gsc_impressions"].shift(-1)
valid_future = (df["future_date"] == df["report_date"] + pd.Timedelta(days=1))
df["future_decline"] = np.nan
eligible = valid_future & (df["gsc_impressions"] >= 5)
df.loc[eligible, "future_decline"] = (df.loc[eligible, "future_impressions"] <= df.loc[eligible, "gsc_impressions"] * 0.80).astype(int)

model_df = df[df["future_decline"].notna()].copy()
model_df = model_df.sort_values("report_date").reset_index(drop=True)

# Time split
unique_dates = sorted(model_df["report_date"].dropna().unique())
split_date = unique_dates[int(len(unique_dates) * 0.80)]
train_df = model_df[model_df["report_date"] < split_date].copy()
test_df = model_df[model_df["report_date"] >= split_date].copy()

model_features = [
    "gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr",
    "impression_delta", "impression_ratio", "position_delta",
    "impression_delta_2d", "impression_vs_3d_avg"
]

# Fit GBDT
model = HistGradientBoostingClassifier(
    max_iter=500, learning_rate=0.015, max_depth=8,
    min_samples_leaf=15, l2_regularization=1.5, random_state=42, class_weight="balanced"
)
model.fit(train_df[model_features], train_df["future_decline"])

# Generate Action Queue
test_df["decline_probability"] = model.predict_proba(test_df[model_features])[:, 1]

# Reason Code Logic
def assign_reason_code(row):
    if row["position_delta"] > 2.0:
        return "POS_SLIP"
    elif row["impression_vs_3d_avg"] < 0.75:
        return "IMP_DROP_VELOCITY"
    elif row["ctr"] < 0.01 and row["gsc_impressions"] >= 20:
        return "LOW_CTR_HIGH_IMP"
    else:
        return "DECAY_RISK"

test_df["reason_code"] = test_df.apply(assign_reason_code, axis=1)
test_df["action"] = np.where(test_df["decline_probability"] >= 0.50, "Review Content", "Monitor")

action_queue = test_df.sort_values("decline_probability", ascending=False).reset_index(drop=True)
action_queue["rank"] = np.arange(len(action_queue)) + 1

print("Action queue successfully generated.")
display(action_queue[[
    "rank", "report_date", "gsc_impressions", "gsc_avg_position",
    "ctr", "decline_probability", "reason_code", "action"
]].head(10))

Imports completed.


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Action queue successfully generated.


,rank,report_date,gsc_impressions,gsc_avg_position,ctr,decline_probability,reason_code,action
0,1,2025-02-22,60,29.716667,0.0,0.663058,LOW_CTR_HIGH_IMP,Review Content
1,2,2025-02-24,12,25.166667,0.0,0.658436,DECAY_RISK,Review Content
2,3,2025-02-24,10,17.100000,0.0,0.657931,DECAY_RISK,Review Content
3,4,2025-02-24,9,19.777778,0.0,0.657931,DECAY_RISK,Review Content
4,5,2025-02-24,16,18.562500,0.0,0.657931,DECAY_RISK,Review Content
5,6,2025-02-25,8,16.125000,0.0,0.655283,DECAY_RISK,Review Content
6,7,2025-02-24,21,17.380952,0.0,0.655237,LOW_CTR_HIGH_IMP,Review Content
7,8,2025-02-24,27,16.407407,0.0,0.655237,LOW_CTR_HIGH_IMP,Review Content
8,9,2025-02-24,23,24.043478,0.0,0.654783,LOW_CTR_HIGH_IMP,Review Content
9,10,2025-02-22,29,26.068966,0.0,0.654783,LOW_CTR_HIGH_IMP,Review Content


## 3. Human review + the no-go list

Human Review Checklist (Before Taking Editorial Action):

Search Intent Verification: Confirm if top target queries shifted intent before rewriting content.

SERP Layout Changes: Inspect if Google added a featured snippet or AI Overview displacing organic clicks.

Technical Audit: Verify the page has no indexing blocks, broken links, canonical errors, or slow load times.

Recent Changes: Ensure the page was not already updated within the last 30 days.

The No-Go List (NEVER Automate):

Automated Content Rewrites: Never allow automated AI tools to auto-rewrite or overwrite published live content without human editor review.

Auto-Redirects or Deletions: Never automatically delete or 301-redirect pages based solely on predicted decline scores.

Bulk URL Slug Changes: Never automate URL pattern changes or canonical shifts based on statistical prioritization queues.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Generate Human Review Verification Flag Summary
action_queue["requires_manual_check"] = np.where(
    action_queue["action"] == "Review Content",
    "SERP Intent, Tech Health, Recent Refresh Date",
    "None (Monitor Only)"
)

review_candidates = action_queue[action_queue["action"] == "Review Content"].copy()
print(f"Total pages requiring human review in test set: {len(review_candidates)} / {len(action_queue)}")
print("Sample review items with assigned human verification checklist:")
display(review_candidates[[
    "rank", "gsc_impressions", "gsc_avg_position", "decline_probability",
    "reason_code", "requires_manual_check"
]].head(5))

Total pages requiring human review in test set: 4748 / 12696
Sample review items with assigned human verification checklist:


,rank,gsc_impressions,gsc_avg_position,decline_probability,reason_code,requires_manual_check
0,1,60,29.716667,0.663058,LOW_CTR_HIGH_IMP,"SERP Intent, Tech Health, Recent Refresh Date"
1,2,12,25.166667,0.658436,DECAY_RISK,"SERP Intent, Tech Health, Recent Refresh Date"
2,3,10,17.100000,0.657931,DECAY_RISK,"SERP Intent, Tech Health, Recent Refresh Date"
3,4,9,19.777778,0.657931,DECAY_RISK,"SERP Intent, Tech Health, Recent Refresh Date"
4,5,16,18.562500,0.657931,DECAY_RISK,"SERP Intent, Tech Health, Recent Refresh Date"


## 4. Monitoring / retrain triggers

The model's ranking ability degrades over time as search dynamics evolve. The pipeline requires retraining when any of the following triggers are breached:Precision Decay (Precision@20 Drop): If the top-20 review queue precision drops below 0.30 (30%) on fresh weekly evaluation batches.Distribution Shift (Population Stability Index / Base Rate Shift): If the test period future_decline target rate moves more than $\pm 10\%$ from the training baseline ($\sim 37\%$).Feature Drift: If average search positions or impression distributions shift significantly following a major search engine core update.Cadence Trigger: Retrain automatically every 30 days on fresh rolling warehouse data.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Monitor baseline drift indicators
train_base = train_df["future_decline"].mean()
test_base = test_df["future_decline"].mean()
base_rate_drift = abs(train_base - test_base)

p20_score = action_queue.head(20)["future_decline"].mean()

monitoring_health = {
    "train_base_rate": float(train_base),
    "test_base_rate": float(test_base),
    "base_rate_drift_abs": float(base_rate_drift),
    "current_p20_precision": float(p20_score),
    "retrain_status": "OK" if p20_score >= 0.30 and base_rate_drift < 0.10 else "TRIGGER_RETRAIN"
}

print("Model Health & Retrain Monitor Status:")
print(json.dumps(monitoring_health, indent=2))


Model Health & Retrain Monitor Status:
{
  "train_base_rate": 0.40058400108651365,
  "test_base_rate": 0.31167296786389415,
  "base_rate_drift_abs": 0.0889110332226195,
  "current_p20_precision": 0.35,
  "retrain_status": "OK"
}


## 5. Exports for the paper

Exporting the decision queue, top actionable review recommendations, and playbook metadata receipt to work/outputs/ to support research paper synthesis and downstream reporting.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Save Action Playbook outputs to work/outputs/
os.makedirs("work/outputs", exist_ok=True)

# 1. Export top 50 review queue
queue_export = action_queue[[
    "rank", "report_date", "client_hash_id", "content_hash_id",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position", "ctr",
    "decline_probability", "reason_code", "action"
]].head(50)

queue_export.to_csv("work/outputs/ml10_action_queue_top50.csv", index=False)

# 2. Export playbook metadata receipt
playbook_receipt = {
    "random_seed": 42,
    "working_slice_rows": len(df),
    "test_set_size": len(test_df),
    "queue_review_recommended_count": int((test_df["action"] == "Review Content").sum()),
    "top_20_precision": float(p20_score),
    "reason_code_counts": test_df[test_df["action"] == "Review Content"]["reason_code"].value_counts().to_dict(),
    "retrain_monitoring": monitoring_health
}

with open("work/outputs/ml10_playbook_receipt.json", "w") as f:
    json.dump(playbook_receipt, f, indent=2)

print("ML-10 Action Playbook exports successfully saved:")
print(" - work/outputs/ml10_action_queue_top50.csv")
print(" - work/outputs/ml10_playbook_receipt.json\n")
print(json.dumps(playbook_receipt, indent=2))

ML-10 Action Playbook exports successfully saved:
 - work/outputs/ml10_action_queue_top50.csv
 - work/outputs/ml10_playbook_receipt.json

{
  "random_seed": 42,
  "working_slice_rows": 50000,
  "test_set_size": 12696,
  "queue_review_recommended_count": 4748,
  "top_20_precision": 0.35,
  "reason_code_counts": {
    "DECAY_RISK": 1943,
    "POS_SLIP": 1713,
    "LOW_CTR_HIGH_IMP": 1029,
    "IMP_DROP_VELOCITY": 63
  },
  "retrain_monitoring": {
    "train_base_rate": 0.40058400108651365,
    "test_base_rate": 0.31167296786389415,
    "base_rate_drift_abs": 0.0889110332226195,
    "current_p20_precision": 0.35,
    "retrain_status": "OK"
  }
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.